# LearnMateAI — Qwen 2.5 LoRA Fine-Tuning

Colab-ready notebook for **LoRA/PEFT** fine-tuning (not full fine-tuning) on Stage 3 dataset output.

**Runtime:** Runtime → Change runtime type → GPU (T4 is enough for 1.5B / 3B with QLoRA).

**Budget note (~USD 45/mo ops):** Prefer `Qwen/Qwen2.5-1.5B-Instruct` + QLoRA on free/cheap Colab. Larger bases only if a sponsored GPU is available.

**Status:** Template is complete and loadable. A full GPU training run has **not** been executed in this repository (see top-level `model-Thevindu/README.md`). Smoke-dataset paths below point at `sample_data/` copied from `lm-legal-smoke-v1`.

## 0 — Install dependencies (Colab)

In [ ]:
# Colab: uncomment. Local GPU: install from requirements.txt instead.
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q transformers==4.44.2 datasets==2.21.0 peft==0.12.0 \
        accelerate==0.33.0 bitsandbytes==0.43.3 trl==0.9.6 \
        sentencepiece protobuf einops
print("IN_COLAB =", IN_COLAB)

## 1 — CONFIG (single source of truth)

Change hyperparameters **only here**. The run-record cell reads this dict so every saved adapter is reproducible.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

CONFIG = {
    # --- Identity ---
    "run_id": f"qwen25-lora-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
    "project": "LearnMateAI",
    "track": "model-Thevindu",

    # --- Base model ---
    "base_model_id": "Qwen/Qwen2.5-1.5B-Instruct",  # upgrade to 3B/7B only with GPU budget
    "torch_dtype": "bfloat16",  # fallback to float16 on older GPUs
    "use_qlora": True,          # 4-bit; set False for full LoRA in bf16 if VRAM allows

    # --- Dataset (Stage 3 output) ---
    "dataset_version": "lm-legal-smoke-v1",
    "train_path": "sample_data/train.jsonl",  # Colab: upload or mount Drive path
    "val_path": "sample_data/val.jsonl",
    "max_seq_length": 1024,
    "packing": False,

    # --- LoRA / PEFT ---
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "bias": "none",
        "task_type": "CAUSAL_LM",
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    },

    # --- Training ---
    "training": {
        "num_train_epochs": 2,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "weight_decay": 0.01,
        "logging_steps": 5,
        "eval_strategy": "steps",
        "eval_steps": 25,
        "save_strategy": "steps",
        "save_steps": 25,
        "save_total_limit": 2,
        "fp16": False,
        "bf16": True,
        "optim": "paged_adamw_8bit",
        "report_to": "none",
        "seed": 42,
    },

    # --- Outputs ---
    "output_dir": "adapters",
    "run_records_dir": "run_records",
}

ADAPTER_DIR = Path(CONFIG["output_dir"]) / CONFIG["run_id"]
RUN_RECORD_PATH = Path(CONFIG["run_records_dir"]) / f"{CONFIG['run_id']}.json"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
Path(CONFIG["run_records_dir"]).mkdir(parents=True, exist_ok=True)

print("run_id       :", CONFIG["run_id"])
print("base_model   :", CONFIG["base_model_id"])
print("dataset      :", CONFIG["dataset_version"])
print("adapter_dir  :", ADAPTER_DIR)
print("run_record   :", RUN_RECORD_PATH)

## 2 — Load Stage 3 JSONL and format for chat SFT

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path: str):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(CONFIG["train_path"])
val_rows = load_jsonl(CONFIG["val_path"])

assert train_rows, f"Empty train set: {CONFIG['train_path']}"
assert all("messages" in r for r in train_rows), "Expected chat `messages` field from Stage 3"

# Confirm dataset_version lineage
versions = {r.get("dataset_version") for r in train_rows}
print(f"train={len(train_rows)}  val={len(val_rows)}  dataset_versions={versions}")
if CONFIG["dataset_version"] not in versions:
    print("WARNING: CONFIG dataset_version does not match records — update CONFIG before a real run.")

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
train_ds[0]["messages"][:2]

## 3 — Tokenizer + base model (QLoRA or LoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["base_model_id"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if CONFIG["torch_dtype"] == "bfloat16" else torch.float16
if not torch.cuda.is_available():
    raise RuntimeError("GPU required for this notebook. Switch Colab runtime to GPU.")

quant_cfg = None
if CONFIG["use_qlora"]:
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model_id"],
    quantization_config=quant_cfg,
    device_map="auto",
    torch_dtype=dtype if not CONFIG["use_qlora"] else None,
    trust_remote_code=True,
)
model.config.use_cache = False
print("loaded", CONFIG["base_model_id"], "qlora=" + str(CONFIG["use_qlora"]))

## 4 — Attach LoRA adapters (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if CONFIG["use_qlora"]:
    model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(**CONFIG["lora"])
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 5 — Train with TRL SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_func(example):
    # Qwen chat template applied by tokenizer
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

t = CONFIG["training"]
# Older GPUs may not support bf16
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
fp16 = not bf16_ok
bf16 = bf16_ok and t["bf16"]

training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / "checkpoints"),
    num_train_epochs=t["num_train_epochs"],
    per_device_train_batch_size=t["per_device_train_batch_size"],
    per_device_eval_batch_size=t["per_device_eval_batch_size"],
    gradient_accumulation_steps=t["gradient_accumulation_steps"],
    learning_rate=t["learning_rate"],
    lr_scheduler_type=t["lr_scheduler_type"],
    warmup_ratio=t["warmup_ratio"],
    weight_decay=t["weight_decay"],
    logging_steps=t["logging_steps"],
    eval_strategy=t["eval_strategy"],
    eval_steps=t["eval_steps"],
    save_strategy=t["save_strategy"],
    save_steps=t["save_steps"],
    save_total_limit=t["save_total_limit"],
    fp16=fp16,
    bf16=bf16,
    optim=t["optim"],
    report_to=t["report_to"],
    seed=t["seed"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

train_result = trainer.train()
FINAL_TRAIN_LOSS = float(train_result.training_loss)
print("final train loss:", FINAL_TRAIN_LOSS)

eval_metrics = trainer.evaluate()
FINAL_EVAL_LOSS = float(eval_metrics.get("eval_loss", float("nan")))
print("final eval loss:", FINAL_EVAL_LOSS)

## 6 — Save adapter weights

In [ ]:
adapter_save_path = ADAPTER_DIR / "adapter"
trainer.model.save_pretrained(str(adapter_save_path))
tokenizer.save_pretrained(str(adapter_save_path))
print("adapter saved to", adapter_save_path)

## 7 — MANDATORY RUN RECORD

Writes hyperparameters, dataset version, and final loss next to the adapter.  
**Do not skip this cell.** An adapter without a run-record is not eligible for evaluation/promotion.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

# Fail loudly if training metrics were never produced
assert "FINAL_TRAIN_LOSS" in dir() or "FINAL_TRAIN_LOSS" in globals(), (
    "FINAL_TRAIN_LOSS missing — run the training cell before writing a run-record."
)

run_record = {
    "run_id": CONFIG["run_id"],
    "project": CONFIG["project"],
    "track": CONFIG["track"],
    "completed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "base_model_id": CONFIG["base_model_id"],
    "dataset_version": CONFIG["dataset_version"],
    "train_path": CONFIG["train_path"],
    "val_path": CONFIG["val_path"],
    "train_examples": len(train_rows),
    "val_examples": len(val_rows),
    "use_qlora": CONFIG["use_qlora"],
    "lora": CONFIG["lora"],
    "training": CONFIG["training"],
    "max_seq_length": CONFIG["max_seq_length"],
    "final_train_loss": FINAL_TRAIN_LOSS,
    "final_eval_loss": FINAL_EVAL_LOSS if "FINAL_EVAL_LOSS" in dir() or "FINAL_EVAL_LOSS" in globals() else None,
    "adapter_path": str(adapter_save_path),
    "status": "completed",
    "notes": "LoRA/PEFT only — base weights not modified. App must keep Gemini (or other API) fallback if adapter unavailable.",
}

# Persist beside adapter AND in run_records/
sidecar = Path(adapter_save_path) / "run_record.json"
for path in (RUN_RECORD_PATH, sidecar):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(run_record, f, indent=2)
    print("wrote", path)

run_record

## Next steps

1. Download `adapters/<run_id>/` (includes `adapter/` + `run_record.json`).
2. Log cost/duration in `04_docs/training_run_log.md`.
3. Evaluate with `03_testing_and_versioning/evaluate_candidate.ipynb` — do **not** promote without passing acceptance thresholds.